```python
# @title Cell section A (markdown text) -- Usage Notes
# ============================================================
```

# Google Colab Skeleton With Countdown Timer GUI
[v4_0_6]

## Installation
1. Copy the complete JSON object that contains this notebook into a plain-text file and save it with the `.ipynb` extension, for example `skeleton.ipynb`.
2. Open https://colab.research.google.com, choose **File → Upload notebook**, and select that file. You can also put the file in Google Drive and open it with Colab.
3. No `pip` installs are needed. `ipywidgets` and `google.colab` come preinstalled.
4. Optional: to use a secret, click the **key icon** (Secrets) in the left sidebar, add a secret named `HYPOTHETICAL_API_KEY` (or any name listed in `SECRET_NAMES_ALL`), and turn on **Notebook access**. A missing optional secret only produces a note in the log.
5. Choose **Runtime → Run all**.

## Cells
| #   | Cell name     | Cell type   | Purpose |
|-----|---------------|-------------|---------|
| A   | Usage notes   | text        | This top level cell containing explanatory text. |
| A-1 | Configuration | Python code | All settings. Secrets are loaded from Colab Secrets and never printed. |
| A-2 | GUI bridge + status log | Python code | The thread-safe update queue and the logger behind the `status` tab. |
| A-3 | Storage       | Python code | Optional Drive mount, the working directory, and the optional log file. |
| A-4 | GUI           | Python code | The `timer` and `status` tabs, the countdown timer, the GUI pump, and the heartbeat display. |

<br>

## Configuration
### Settings
- Cell A-1, the first of the code cells, defines configuation settings.

### Using `Colab Secrets` to preserve and hide settings
- Read a secret without printing it with `get_secret('HYPOTHETICAL_API_KEY')`.

<br>

## Usage Notes
### `timer` tab
- Demonstrates a countdown timer starting at 10 and decreases by 1 each second.
- Shows the seconds remaining.
- At zero the countdown stops and the display shows **EXPIRED**.
- **Start** begins the countdown, or restarts it, from 10.
- **Stop** halts it.
- **Reset** returns the display to 10 without starting.
- With `COUNTDOWN_AUTO_START = True`, the countdown starts as soon as the GUI appears.

### `status` tab
- Shows timestamped `NOTE`, `WARNING` and `ERROR` lines, one line per event, appended at the bottom.
- The box stays scrolled to the newest line.
- **Clear log** clears only the on-screen box; the log file is not changed.
- Send messages to the log with `STATUS_LOG.note('...')`, `STATUS_LOG.warning('...')` or `STATUS_LOG.error('...')`.

<br>

## How background updates reach the screen (the GUI pump)
Colab appears to deliver widget updates to the browser only while the kernel is handling a request from the browser. A request means a running cell or a widget event such as a button click. Updates made later by a background thread or asyncio task change the value inside the kernel, but they never reach the screen. That is why earlier versions froze at 10 while the log was still correct. The notebook works around this in three steps:

1. **Posting.** Background code never touches widgets directly. It calls `GUI.post(key, function)` to record the update it wants.
2. **Heartbeat.** A browser-side heartbeat sends an event to the kernel every `HEARTBEAT_GUI_PUMP_INTERVAL_MS` milliseconds (default 250). The default source is an `ipywidgets.Play` control that is started automatically by a small script.
3. **Flushing.** Each heartbeat calls `GUI.flush()`. This applies the pending updates while the browser event is being handled, so they do reach the screen.

When the heartbeat row is hidden or simplified, the Play control still exists on the page. It is only hidden with CSS, so the automatic start still works.

**Rule for future features:** code running outside a cell or button handler must call `GUI.post(...)`. It must never set widget properties directly.

<br>

## Notes on creating and referencing GUI widgets in a safe sequence
- **Python builds everything in order.** In `widgets.Tab(children=[timer_tab, status_tab])`, Python evaluates the list first. Each child widget, and every widget inside it, is fully constructed before the Tab is created. Nothing is built in the background, so there is no moment when the Tab can refer to a child that is not ready. A slow child simply delays the line that creates the Tab.
- **The browser side is handled by ipywidgets.** The browser receives the widget models as messages and renders them asynchronously. Children are referenced by model ID, and the ipywidgets widget manager waits until every referenced child model is available before it draws the Tab. This is true whether the children are passed at construction or assigned later.

**What this skeleton does:**
- It builds every tab's contents first.
- It then creates the Tab through a small helper, `build_tabs([(title, content), ...])`. The helper sets the titles only after the children exist, which is correct for both ipywidgets 7 and 8.
- It calls `display()` only after the whole widget tree is complete.

**Where the real timing hazard lies.** It is in updating widgets from background threads after display, and the GUI pump covers that. If a future tab needs slow work (files, network, API calls), build a lightweight placeholder, display it, and fill it in later through `GUI.post(...)`.

<br>

## Troubleshooting
- **The heartbeat does not move:** this applies when the `Heartbeat (x)` character never changes, or the count stays at `waiting`.
  1. Set `HEARTBEAT_UI_SIMPLIFIED = False` and re-run all cells.
  2. Press the play (▶) button on the player control once.
  3. If that still fails, set `HEARTBEAT_GUI_PUMP_BACKEND = 'js'` and re-run all cells. This uses a JavaScript timer calling `google.colab.kernel.invokeFunction` instead of the Play control.
- **The heartbeat moves but the timer tab does not change:** please report it. That would disprove the diagnosis above.
- **Background browser tabs:** browsers slow down timers in tabs that are not in front, so the display may refresh only about once per second there.
- **Widgets do not render at all:** choose **Runtime → Restart session**, then **Run all**.
- **After changing configuration:** re-run all cells below cell 1, in order.



In [ ]:
# @title Cell A-1 (Python code) -- CONFIGURATION  (edit values in this cell only)
# ============================================================

# ---- File storage ----
FILES_USE_GOOGLE_DRIVE = False      # True: mount Drive and work in FILES_DRIVE_WORKING_DIR; False: use FILES_LOCAL_WORKING_DIR
FILES_DRIVE_MOUNT_POINT = '/content/drive'
FILES_DRIVE_WORKING_DIR = '/content/drive/My Drive/tools/colab/skeleton'
FILES_FORCE_REMOUNT_DRIVE = False
FILES_LOCAL_WORKING_DIR = '/content'   # Colab's default temporary storage (lost when the runtime is recycled)

# ---- Countdown timer ----
COUNTDOWN_FULL_COUNT_SEC = 10       # starting value in seconds
COUNTDOWN_TICK_INTERVAL_SEC = 1.0   # seconds between decrements
COUNTDOWN_EXPIRED_TEXT = 'EXPIRED'  # shown in place of the number when it reaches zero
COUNTDOWN_AUTO_START = False        # True: start counting as soon as the GUI appears

# ---- Heartbeat backend ----
HEARTBEAT_TIMER_BACKEND = 'thread'      # 'thread' (recommended) or 'asyncio'

# ---- Heartbeat GUI pump (browser heartbeat that lets background updates reach the screen) ----
HEARTBEAT_GUI_PUMP_BACKEND = 'play'     # 'play' (ipywidgets Play control, recommended) or 'js' (JavaScript timer)
HEARTBEAT_GUI_PUMP_INTERVAL_MS = 250    # how often pending GUI updates are painted

# ---- Heartbeat display (row below the tabs) ----
HEARTBEAT_UI_VISIBLE = True      # False hides the heartbeat row entirely (the pump keeps running)
HEARTBEAT_UI_SIMPLIFIED = True   # True: 'Heartbeat (-)' spinner only; False: play control + heartbeat count

# ---- Status log ----
LOG_TIMEZONE = 'UTC'          # e.g. 'America/New_York'
LOG_MAX_LINES = 1000          # oldest lines are dropped from the display beyond this
LOG_BOX_HEIGHT = '320px'
LOG_TO_FILE = True            # also append log lines to a file in the working directory
LOG_FILE_NAME = 'status.log'

# ---- Secrets (Colab key icon in the left sidebar) ----
SECRET_NAMES_ALL = ['HYPOTHETICAL_API_KEY']   # names to try to load
SECRET_NAMES_REQUIRED = []             # subset of SECRET_NAMES_ALL that must exist (missing -> ERROR)

# ------------------------------------------------------------
# Secret loading - values are never printed or logged.
# ------------------------------------------------------------
SECRET_LOADED_KV_PAIRS = {}
CONFIG_MESSAGES = []   # (level, text) pairs, replayed into the status log once it exists


def _load_secrets(names, required):
    try:
        from google.colab import userdata
    except ImportError:
        CONFIG_MESSAGES.append(('WARNING', 'google.colab.userdata is unavailable (not running in Colab?); no secrets loaded.'))
        return
    for name in names:
        try:
            value = userdata.get(name)
        except Exception as exc:  # SecretNotFoundError, NotebookAccessError, ...
            level = 'ERROR' if name in required else 'NOTE'
            CONFIG_MESSAGES.append((level, f'Secret {name!r} not loaded ({type(exc).__name__}).'))
            continue
        if value:
            SECRET_LOADED_KV_PAIRS[name] = value
            CONFIG_MESSAGES.append(('NOTE', f'Secret {name!r} loaded ({len(value)} characters).'))
        else:
            level = 'ERROR' if name in required else 'WARNING'
            CONFIG_MESSAGES.append((level, f'Secret {name!r} exists but is empty.'))


def get_secret(name, default=None):
    """Return a loaded secret value (or default) without exposing it in output."""
    return SECRET_LOADED_KV_PAIRS.get(name, default)


if HEARTBEAT_TIMER_BACKEND not in ('thread', 'asyncio'):
    CONFIG_MESSAGES.append(('WARNING', f'Unknown HEARTBEAT_TIMER_BACKEND {HEARTBEAT_TIMER_BACKEND!r}; using thread.'))
    HEARTBEAT_TIMER_BACKEND = 'thread'
if HEARTBEAT_GUI_PUMP_BACKEND not in ('play', 'js'):
    CONFIG_MESSAGES.append(('WARNING', f'Unknown HEARTBEAT_GUI_PUMP_BACKEND {HEARTBEAT_GUI_PUMP_BACKEND!r}; using play.'))
    HEARTBEAT_GUI_PUMP_BACKEND = 'play'

_load_secrets(SECRET_NAMES_ALL, SECRET_NAMES_REQUIRED)
for _level, _text in CONFIG_MESSAGES:
    print(f'{_level:<7} {_text}')
print('Configuration loaded.')


In [ ]:
# @title Cell A-2 (Python code) -- GUI BRIDGE + STATUS LOG
# ============================================================

import datetime
import html
import threading

import ipywidgets as widgets

try:
    from zoneinfo import ZoneInfo
    _LOG_TZ = ZoneInfo(LOG_TIMEZONE)
except Exception:
    _LOG_TZ = datetime.timezone.utc


class GuiBridge:
    """Thread-safe hand-off of GUI updates.

    Any thread may call post(key, fn). Nothing touches a widget until flush() runs, and flush()
    must only be called while the kernel is handling a browser request (a running cell, a button
    click, or a GUI pump heartbeat), because only then does Colab deliver widget updates.
    A newer post with the same key replaces an older one that has not been painted yet.
    """

    def __init__(self):
        self._lock = threading.Lock()
        self._pending = {}
        self.error_handler = None

    def post(self, key, fn):
        with self._lock:
            self._pending.pop(key, None)
            self._pending[key] = fn

    def flush(self):
        with self._lock:
            jobs = list(self._pending.items())
            self._pending.clear()
        for key, fn in jobs:
            try:
                fn()
            except Exception as exc:
                if self.error_handler is not None:
                    self.error_handler(f'GUI update {key!r} failed: {type(exc).__name__}: {exc}')
        return len(jobs)


class StatusLog:
    """Collects timestamped NOTE / WARNING / ERROR lines; safe to call from any thread.

    The HTML for the log box is built immediately, but it is painted through the GuiBridge.
    """

    LEVEL_COLORS = {'NOTE': '#1a7f37', 'WARNING': '#9a6700', 'ERROR': '#cf222e'}

    def __init__(self, gui, max_lines=1000, height='320px'):
        self._gui = gui
        self._lock = threading.RLock()
        self._entries = []
        self._max_lines = max(1, int(max_lines))
        self._height = height
        self._log_path = None
        self.widget = widgets.HTML(layout=widgets.Layout(width='100%'))
        self._render()

    # ---- public API ----
    def note(self, message):
        self.log('NOTE', message)

    def warning(self, message):
        self.log('WARNING', message)

    def error(self, message):
        self.log('ERROR', message)

    def log(self, level, message):
        level = str(level).upper()
        if level not in self.LEVEL_COLORS:
            level = 'NOTE'
        entry = (self._now(), level, str(message))
        with self._lock:
            self._entries.append(entry)
            if len(self._entries) > self._max_lines:
                del self._entries[:-self._max_lines]
            self._append_to_file([entry])
            self._render()

    def clear(self):
        """Clear the on-screen log (the log file, if any, is not touched)."""
        with self._lock:
            self._entries = []
            self._render()

    def set_log_file(self, path):
        """Start mirroring log lines to a file (entries logged so far are written first)."""
        with self._lock:
            self._log_path = path
            self._append_to_file(list(self._entries))
            self._render()

    @property
    def count(self):
        with self._lock:
            return len(self._entries)

    # ---- internals ----
    @staticmethod
    def _now():
        return datetime.datetime.now(_LOG_TZ).strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]

    @staticmethod
    def _plain(entry):
        stamp, level, text = entry
        return f'[{stamp}] {level:<7} {text}'

    def _append_to_file(self, entries):
        if not self._log_path or not entries:
            return
        try:
            with open(self._log_path, 'a', encoding='utf-8') as fh:
                for entry in entries:
                    fh.write(self._plain(entry) + '\n')
        except Exception as exc:
            failed_path, self._log_path = self._log_path, None   # stop retrying
            self._entries.append((self._now(), 'ERROR',
                                  f'Log file {failed_path} disabled: {type(exc).__name__}: {exc}'))

    def _render(self):
        rows = []
        for stamp, level, text in self._entries:
            color = self.LEVEL_COLORS[level]
            rows.append(
                '<div style="white-space:pre-wrap; word-break:break-word;">'
                f'<span style="color:#57606a;">[{stamp}]</span> '
                f'<b style="color:{color};">{level:<7}</b> '
                f'{html.escape(text)}</div>'
            )
        if not rows:
            rows.append('<div style="color:#8c959f;">(no log entries)</div>')
        # flex-direction:column-reverse keeps the scroll position pinned to the newest (bottom) line
        value = (
            f'<div style="height:{self._height}; overflow-y:auto; display:flex; '
            'flex-direction:column-reverse; border:1px solid #d0d7de; border-radius:6px; '
            'padding:6px 8px; background:#ffffff; color:#24292f; '
            'font-family:monospace; font-size:12px; line-height:1.45;">'
            '<div>' + ''.join(rows) + '</div></div>'
        )
        widget = self.widget

        def apply():
            widget.value = value

        self._gui.post('status_log', apply)


GUI = GuiBridge()
STATUS_LOG = StatusLog(GUI, max_lines=LOG_MAX_LINES, height=LOG_BOX_HEIGHT)
GUI.error_handler = STATUS_LOG.error
STATUS_LOG.note('Status log initialized.')
for _level, _text in CONFIG_MESSAGES:
    STATUS_LOG.log(_level, f'[config] {_text}')
GUI.flush()
print(f'Status log ready ({STATUS_LOG.count} entries). It is displayed under the "status" tab.')


In [ ]:
# @title Cell A-3 (Python code) -- STORAGE: optional Google Drive mount + working directory
# ============================================================
import os

WORKING_DIR = FILES_LOCAL_WORKING_DIR

if FILES_USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(FILES_DRIVE_MOUNT_POINT, force_remount=FILES_FORCE_REMOUNT_DRIVE)
        STATUS_LOG.note(f'Google Drive mounted at {FILES_DRIVE_MOUNT_POINT}.')
        os.makedirs(FILES_DRIVE_WORKING_DIR, exist_ok=True)
        WORKING_DIR = FILES_DRIVE_WORKING_DIR
    except Exception as exc:
        STATUS_LOG.error(f'Google Drive setup failed: {type(exc).__name__}: {exc}')
        STATUS_LOG.warning(f'Falling back to temporary storage at {FILES_LOCAL_WORKING_DIR}.')
else:
    STATUS_LOG.note('Google Drive mounting is disabled (FILES_USE_GOOGLE_DRIVE = False); '
                    f'using temporary storage at {FILES_LOCAL_WORKING_DIR}.')

try:
    os.makedirs(WORKING_DIR, exist_ok=True)
    os.chdir(WORKING_DIR)
    STATUS_LOG.note(f'Working directory set to {os.getcwd()}')
    if LOG_TO_FILE:
        _log_path = os.path.join(WORKING_DIR, LOG_FILE_NAME)
        STATUS_LOG.set_log_file(_log_path)
        STATUS_LOG.note(f'Log lines are also appended to {_log_path}')
except Exception as exc:
    STATUS_LOG.error(f'Could not use working directory {WORKING_DIR}: {type(exc).__name__}: {exc}')

GUI.flush()
print('Current working directory:', os.getcwd())


In [ ]:
# @title Cell A-4 (Python code) -- GUI: tabs 'timer' and 'status', countdown timer, GUI pump, heartbeat display
# ============================================================
import asyncio
import html
import os
import threading
import time

import ipywidgets as widgets
from IPython.display import display, Javascript

# If this cell is re-run, shut down what the previous run left behind.
try:
    COUNTDOWN_TIMER.stop(silent=True)
except NameError:
    pass
try:
    HEARTBEAT_GUI_PUMP_PLAY.close()
except NameError:
    pass


class CountdownTimer:
    """Counts down from start_value once per interval; reports via on_update(remaining, expired, running).

    on_update may be called from a background thread, so it must not touch widgets directly;
    it should post to the GuiBridge instead.
    Every start() gets a new generation number. A worker only applies ticks while its generation is
    current, so stop/reset/restart can never be corrupted by a late tick from an old worker.
    """

    def __init__(self, start_value, interval, on_update, log, backend='thread'):
        self._start_value = int(start_value)
        self._interval = float(interval)
        self._on_update = on_update
        self._log = log
        self._backend = backend
        self._lock = threading.Lock()
        self._generation = 0
        self._remaining = self._start_value
        self._running = False
        self._stop_event = None
        self._task = None

    @property
    def running(self):
        with self._lock:
            return self._running

    @property
    def remaining(self):
        with self._lock:
            return self._remaining

    # ---- control ----
    def start(self):
        with self._lock:
            already_running = self._running
            if not already_running:
                self._generation += 1
                gen = self._generation
                self._remaining = self._start_value
                self._running = True
                self._stop_event = threading.Event()
                stop_event = self._stop_event
        if already_running:
            self._log.warning('Start ignored: the countdown is already running.')
            return
        self._on_update(self._start_value, False, True)
        self._log.note(f'Countdown started at {self._start_value} s (backend: {self._backend}).')
        if self._backend == 'asyncio':
            self._task = asyncio.ensure_future(self._run_async(gen))
        else:
            worker = threading.Thread(target=self._run_thread, args=(gen, stop_event),
                                      name=f'heartbeat-countdown-{gen}', daemon=True)
            worker.start()

    def stop(self, silent=False):
        with self._lock:
            was_running = self._running
            self._running = False
            self._generation += 1          # invalidates any in-flight worker
            stop_event, self._stop_event = self._stop_event, None
            task, self._task = self._task, None
            remaining = self._remaining
        if stop_event is not None:
            stop_event.set()
        if task is not None and not task.done():
            task.cancel()
        if not silent:
            if was_running:
                self._on_update(remaining, False, False)
                self._log.note(f'Countdown stopped by user with {remaining} s remaining.')
            else:
                self._log.warning('Stop ignored: the countdown is not running.')
        return was_running

    def reset(self):
        self.stop(silent=True)
        with self._lock:
            self._remaining = self._start_value
        self._on_update(self._start_value, False, False)
        self._log.note(f'Countdown reset to {self._start_value} s.')

    # ---- tick handling (shared by both backends) ----
    def _handle_tick(self, gen):
        """Apply one decrement. Returns True if the worker should keep going."""
        with self._lock:
            if gen != self._generation or not self._running:
                return False               # stale worker: stopped, reset or restarted
            self._remaining -= 1
            remaining = self._remaining
            expired = remaining <= 0
            if expired:
                self._remaining = 0
                self._running = False
        self._on_update(remaining, expired, not expired)
        if expired:
            self._log.note('Countdown reached zero: EXPIRED. Timer stopped.')
        return not expired

    def _fail(self, gen, exc):
        with self._lock:
            if gen == self._generation:
                self._running = False
        self._log.error(f'Countdown worker failed: {type(exc).__name__}: {exc}')

    # ---- backend: background thread ----
    def _run_thread(self, gen, stop_event):
        try:
            next_deadline = time.monotonic() + self._interval
            while True:
                delay = max(0.0, next_deadline - time.monotonic())
                if stop_event.wait(delay):
                    return                 # stop requested
                next_deadline += self._interval   # deadline-based: no cumulative drift
                if not self._handle_tick(gen):
                    return
        except Exception as exc:
            self._fail(gen, exc)

    # ---- backend: asyncio task on the kernel event loop ----
    async def _run_async(self, gen):
        loop = asyncio.get_event_loop()
        try:
            next_deadline = loop.time() + self._interval
            while True:
                await asyncio.sleep(max(0.0, next_deadline - loop.time()))
                next_deadline += self._interval
                if not self._handle_tick(gen):
                    return
        except asyncio.CancelledError:
            return
        except Exception as exc:
            self._fail(gen, exc)


def build_tabs(named_children):
    """Create a Tab from fully built (title, widget) pairs.

    Children are passed at construction and titles are set afterwards, which is the order that
    works in both ipywidgets 7 and 8 (ipywidgets 8 trims titles to the number of children).
    """
    tab = widgets.Tab(children=[child for _title, child in named_children])
    for index, (title, _child) in enumerate(named_children):
        tab.set_title(index, title)
    return tab


# ---------------- widgets ----------------
TIMER_DIGITS = widgets.HTML()
TIMER_STATE = widgets.HTML()
HEARTBEAT_GUI_HTML = widgets.HTML()
start_button = widgets.Button(description='Start', button_style='success', icon='play')
stop_button = widgets.Button(description='Stop', button_style='danger', icon='stop')
reset_button = widgets.Button(description='Reset', button_style='', icon='refresh')
clear_log_button = widgets.Button(description='Clear log', icon='trash')


def render_timer(remaining, expired, running):
    """Describe how the timer tab should look and post it to the GuiBridge (any thread)."""
    if expired:
        big, color, state = COUNTDOWN_EXPIRED_TEXT, '#cf222e', 'Countdown finished. Press Start to run it again.'
    else:
        big = str(remaining)
        color = '#0969da' if running else '#57606a'
        state = 'Running...' if running else 'Stopped. Press Start to begin.'
    digits_html = (
        '<div style="font-family:monospace; font-size:72px; font-weight:bold; '
        f'color:{color}; min-width:340px; text-align:center; padding:10px 0;">'
        f'{html.escape(big)}</div>'
    )
    state_html = f'<div style="text-align:center; color:#57606a;">{html.escape(state)}</div>'

    def apply():
        TIMER_DIGITS.value = digits_html
        TIMER_STATE.value = state_html
        start_button.disabled = running
        stop_button.disabled = not running

    GUI.post('timer', apply)


# ---------------- heartbeat display ----------------
SPINNER_CHARS = ('-', '/', '|', '\\')   # dash, forward slash, vertical bar, backslash


def _heartbeat_html(count):
    if HEARTBEAT_UI_SIMPLIFIED:
        char = SPINNER_CHARS[count % len(SPINNER_CHARS)]
        return f'<span style="font-family:monospace; font-size:13px;">Heartbeat ({html.escape(char)})</span>'
    if count == 0:
        text = 'GUI pump: waiting. If this does not change within a few seconds, press the play button on the left.'
        color = '#9a6700'
    else:
        text = f'GUI pump: running (heartbeat {count})'
        color = '#1a7f37'
    return f'<span style="font-size:12px; color:{color};">{html.escape(text)}</span>'


COUNTDOWN_TIMER = CountdownTimer(COUNTDOWN_FULL_COUNT_SEC, COUNTDOWN_TICK_INTERVAL_SEC, render_timer, STATUS_LOG, HEARTBEAT_TIMER_BACKEND)
render_timer(COUNTDOWN_FULL_COUNT_SEC, False, False)
HEARTBEAT_GUI_HTML.value = _heartbeat_html(0)

# ---------------- GUI pump ----------------
HEARTBEAT_GUI_PUMP_STATE = {'count': 0}


def gui_pump(*_args, **_kwargs):
    """Runs once per browser heartbeat, i.e. while the kernel is handling a browser request.

    This is the moment when widget updates reach the screen, so pending updates are applied here.
    """
    try:
        HEARTBEAT_GUI_PUMP_STATE['count'] += 1
        count = HEARTBEAT_GUI_PUMP_STATE['count']
        if count == 1:
            STATUS_LOG.note(f'GUI pump is running (backend: {HEARTBEAT_GUI_PUMP_BACKEND}, every {HEARTBEAT_GUI_PUMP_INTERVAL_MS} ms).')
        if HEARTBEAT_UI_VISIBLE:
            HEARTBEAT_GUI_HTML.value = _heartbeat_html(count)
        GUI.flush()
    except Exception as exc:
        STATUS_LOG.error(f'GUI pump failed: {type(exc).__name__}: {exc}')


# The Play control counts in the browser and reports each step to the kernel.
# It is always placed on the page (so it can run), but it is hidden with CSS unless the
# full diagnostic heartbeat UI is requested.
_show_play = HEARTBEAT_UI_VISIBLE and not HEARTBEAT_UI_SIMPLIFIED
HEARTBEAT_GUI_PUMP_PLAY = widgets.Play(value=0, min=0, max=10**9, step=1, interval=int(HEARTBEAT_GUI_PUMP_INTERVAL_MS),
                         layout=widgets.Layout(width='auto', display=None if _show_play else 'none'))
for _trait in ('repeat', '_repeat'):
    if HEARTBEAT_GUI_PUMP_PLAY.has_trait(_trait):
        setattr(HEARTBEAT_GUI_PUMP_PLAY, _trait, True)
HEARTBEAT_GUI_PUMP_PLAY.add_class('heartbeat-gui-pump-play-control')
HEARTBEAT_GUI_PUMP_PLAY.observe(lambda change: gui_pump(), names='value')


# ---------------- button handlers ----------------
def _safe(action, label):
    """Wrap a button action: log exceptions, then paint pending updates (a click is a browser request)."""
    def handler(_button):
        try:
            action()
        except Exception as exc:
            STATUS_LOG.error(f'{label} failed: {type(exc).__name__}: {exc}')
        finally:
            GUI.flush()
    return handler


def _clear_log():
    STATUS_LOG.clear()
    STATUS_LOG.note('Log display cleared (the log file is unchanged).')


start_button.on_click(_safe(COUNTDOWN_TIMER.start, 'Start'))
stop_button.on_click(_safe(COUNTDOWN_TIMER.stop, 'Stop'))
reset_button.on_click(_safe(COUNTDOWN_TIMER.reset, 'Reset'))
clear_log_button.on_click(_safe(_clear_log, 'Clear log'))

# ---------------- layout (all contents are built before the Tab is created) ----------------
timer_tab = widgets.VBox(
    [
        widgets.HTML('<div style="text-align:center; font-weight:bold;">Seconds remaining</div>'),
        TIMER_DIGITS,
        TIMER_STATE,
        widgets.HBox([start_button, stop_button, reset_button],
                     layout=widgets.Layout(justify_content='center', margin='10px 0 0 0')),
    ],
    layout=widgets.Layout(padding='10px', align_items='center'),
)
status_tab = widgets.VBox(
    [STATUS_LOG.widget, widgets.HBox([clear_log_button], layout=widgets.Layout(margin='6px 0 0 0'))],
    layout=widgets.Layout(padding='10px'),
)

tabs = build_tabs([('timer', timer_tab), ('status', status_tab)])

heartbeat_row = widgets.HBox(
    [HEARTBEAT_GUI_PUMP_PLAY, HEARTBEAT_GUI_HTML],
    layout=widgets.Layout(align_items='center', margin='6px 0 0 0',
                          display=None if HEARTBEAT_UI_VISIBLE else 'none'),
)
app = widgets.VBox([widgets.HTML('<h3 style="margin:4px 0;">Skeleton app</h3>'), tabs, heartbeat_row])

GUI.flush()
display(app)

STATUS_LOG.note(f'GUI displayed (ipywidgets {widgets.__version__}; working dir: {os.getcwd()}; '
                f'heartbeat UI visible={HEARTBEAT_UI_VISIBLE}, simplified={HEARTBEAT_UI_SIMPLIFIED}).')
if COUNTDOWN_AUTO_START:
    COUNTDOWN_TIMER.start()
GUI.flush()

# ---------------- start the browser heartbeat ----------------
if HEARTBEAT_GUI_PUMP_BACKEND == 'js':
    # Alternative pump: a JavaScript timer that calls the kernel directly.
    try:
        from google.colab import output as colab_output
        colab_output.register_callback('heartbeat.gui_pump', gui_pump)
        _js = """
        (function () {
          let busy = false;
          setInterval(async function () {
            if (busy) { return; }
            busy = true;
            try { await google.colab.kernel.invokeFunction('heartbeat.gui_pump', [], {}); }
            catch (e) { }
            busy = false;
          }, __MS__);
        })();
        """.replace('__MS__', str(int(HEARTBEAT_GUI_PUMP_INTERVAL_MS)))
        display(Javascript(_js))
    except Exception as exc:
        STATUS_LOG.error(f'Could not start the js GUI pump: {type(exc).__name__}: {exc}')
        GUI.flush()
else:
    # Default pump: press the (possibly hidden) Play control's play button automatically.
    _js = """
    (function () {
      let tries = 0;
      const iv = setInterval(function () {
        tries += 1;
        const button = document.querySelector('.heartbeat-gui-pump-play-control button');
        if (button) { clearInterval(iv); button.click(); }
        else if (tries > 150) { clearInterval(iv); }
      }, 200);
    })();
    """
    display(Javascript(_js))
